# Data Cleaning Pipeline

Reproducible pandas/Jupyter workflow that transforms a messy e-commerce orders dataset
(CSV with an embedded JSON address column and a free-text notes column) into a
documented, analysis-ready dataset.

This notebook reads top-to-bottom like a report: each cleaning stage has a Markdown
cell explaining the decision, followed by the code that applies it.

## 1. Load and profile (before)

Load the raw dataset exactly as generated and capture its baseline shape, dtypes,
missing values, duplicates, and inconsistent categories. These numbers are the
"before" side of the before/after comparison.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/orders_raw.csv")

In [2]:
df.shape

(203, 10)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          202 non-null    float64
 1   customer_name     203 non-null    str    
 2   email             162 non-null    str    
 3   order_date        203 non-null    str    
 4   product           203 non-null    str    
 5   quantity          170 non-null    float64
 6   unit_price        119 non-null    str    
 7   shipping_address  203 non-null    str    
 8   notes             162 non-null    str    
 9   status            203 non-null    str    
dtypes: float64(2), str(8)
memory usage: 16.0 KB


In [4]:
df.isna().sum()

order_id             1
customer_name        0
email               41
order_date           0
product              0
quantity            33
unit_price          84
shipping_address     0
notes               41
status               0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(2)

In [6]:
df["status"].unique()

<StringArray>
['Cancelled', 'cancelled', 'delivered', 'Delivered', 'Pending', 'DELIVERED']
Length: 6, dtype: str

In [7]:
df["product"].unique()

<StringArray>
['laptop ', 'LAPTOP', 'Laptop', 'Mouse', 'Monitor', 'Keyboard', 'Headphones']
Length: 7, dtype: str

## 2. Clean, one decision at a time

Each decision below is a separate Markdown + code cell so the reasoning is
explicit and reviewable. The order is deliberate: drop structural problems
(duplicates, missing key) first, then normalize values, then expand nested data.

### 2.1 Remove exact duplicate rows

`df.duplicated().sum()` reported 2 exact duplicate rows. Exact duplicates carry no
extra information and would double-count orders, so drop them. Rows: 203 to 201.

In [8]:
df = df.drop_duplicates()

### 2.2 Drop rows with a missing order_id

A row with no `order_id` cannot be reliably tracked or deduplicated later. Rather
than invent a synthetic ID (which would fabricate data), drop the row. With the
missing value gone, cast `order_id` from float back to int. Rows: 201 to 200.

In [9]:
df = df.dropna(subset=["order_id"])
df["order_id"] = df["order_id"].astype(int)

### 2.3 Standardize customer_name casing and whitespace

The same person appears as `Ada Obi`, `ada obi`, `BELLO MUSA`, and `  Efe Okafor  `.
Strip surrounding whitespace and title-case so names are comparable across rows.

In [10]:
df["customer_name"] = df["customer_name"].str.strip().str.title()

### 2.4 Standardize product and status categories

Mixed casing splits one real category into several: `product` contained
`Laptop` / `LAPTOP` / `laptop ` and `status` contained `Delivered` / `delivered` /
`DELIVERED`. Strip and title-case both so each category collapses to one value.

In [11]:
df["product"] = df["product"].str.strip().str.title()
df["status"] = df["status"].str.strip().str.title()

### 2.5 Parse the three mixed order_date formats

The raw `order_date` mixes three layouts in one column - ISO (`2026-01-03`), US
(`01/24/2026`) and day-month name (`15-Feb-2026`). `pd.to_datetime(..., format="mixed")`
(pandas 2.0+) infers the format per value and returns a real datetime column.

In [12]:
df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")

### 2.6 Clean unit_price into a numeric column

Prices mix plain numbers (`199.99`), currency-prefixed strings (`$49.99`), and
missing markers. Normalize the markers, strip `$`, then coerce to numeric. Where a
price is unknown the order value is unknowable, so leave it as a real missing value
(`NaN`) rather than guessing a number.

In [13]:
df["unit_price"] = df["unit_price"].replace({"N/A": None, "": None})
df["unit_price"] = df["unit_price"].astype(str).str.replace("$", "", regex=False)
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

### 2.7 Flag invalid quantities (do not drop)

Negative or zero quantities are logically invalid for an order, and a missing
quantity is unknown. Policy: flag these rows rather than drop them, since the rest
of the row (customer, address, notes) may still be useful for other analysis. This
adds a boolean `quantity_invalid` column; the original `quantity` is kept as-is.

In [14]:
df["quantity_invalid"] = df["quantity"].isna() | (df["quantity"] <= 0)

### 2.8 Parse the embedded JSON shipping_address

`shipping_address` holds a JSON object per row (e.g. city/state/zip). Parse each
string and expand it into real `city`, `state`, and `zip` columns, then drop the
original string column.

In [15]:
import json

address_df = df["shipping_address"].apply(json.loads).apply(pd.Series)
df = pd.concat([df.drop(columns=["shipping_address"]), address_df], axis=1)

### 2.9 Clean the free-text notes column

Notes contain stray internal whitespace and newlines, and invisible empty strings.
Collapse runs of whitespace to a single space, strip the ends, and turn empty strings
into a real missing value so blanks are not mistaken for content.

In [16]:
df["notes"] = df["notes"].str.replace(r"\s+", " ", regex=True).str.strip()
df["notes"] = df["notes"].replace("", pd.NA)

## 3. Profile "after" and compare

Re-run the same profiling commands from section 1 against the cleaned frame so the
before/after numbers line up one-to-one. The written comparison, with the numbers
side by side, lives in `docs/profile_before_after.md`.

In [17]:
df.shape

(200, 13)

In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          200 non-null    int64         
 1   customer_name     200 non-null    str           
 2   email             160 non-null    str           
 3   order_date        200 non-null    datetime64[us]
 4   product           200 non-null    str           
 5   quantity          167 non-null    float64       
 6   unit_price        117 non-null    float64       
 7   notes             161 non-null    str           
 8   status            200 non-null    str           
 9   quantity_invalid  200 non-null    bool          
 10  city              200 non-null    str           
 11  state             200 non-null    str           
 12  zip               200 non-null    str           
dtypes: bool(1), datetime64[us](1), float64(2), int64(1), str(8)
memory usage: 20.5 KB


In [19]:
df.isna().sum()

order_id             0
customer_name        0
email               40
order_date           0
product              0
quantity            33
unit_price          83
notes               39
status               0
quantity_invalid     0
city                 0
state                0
zip                  0
dtype: int64

In [20]:
df.duplicated().sum()

np.int64(0)